## 01 — Data Loading and Inspection

- Load IMU and conditions data for a single subject
- Verify structure and sampling rate against documentation
- Plot raw signals
- Not for official analysis

In [1]:
import scipy.io as sio

conditions_path = "/Volumes/LPM02 storage/Datasets/Bio/Camargo_2021/AB06/10_09_18/stair/conditions/stair_1_l_01_01.mat"

data = sio.loadmat(conditions_path)
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['__header__', '__version__', '__globals__', 'subject', 'file', 'trialStarts', 'trialEnds', 'transLegAscent', 'transLegDescent', 'stairHeight', 'None', '__function_workspace__'])


In [2]:
for key in ['file', 'transLegAscent', 'stairHeight', 'trialStarts', 'trialEnds']:
    print(f"{key}: {data[key]}")

file: ['D:\\Dropbox (GaTech)\\EPIC_DATASETS\\c3dStuff\\AB06\\10_09_18\\stair\\TRC\\stair_1_l_01.mat']
transLegAscent: ['l' 'r']
stairHeight: [[4]]
trialStarts: [[13.79]]
trialEnds: [[29.01]]


Conditions file setup varies by locomotion mode:
- `trialStarts` and `trialEnds` are consistent across modes. 
- Loading conditions data will require mode-aware parsing.

In [3]:
# Print out conditions for each mode
paths = [
    '/Volumes/LPM02 storage/Datasets/Bio/Camargo_2021/AB06/10_09_18/levelground/conditions/levelground_ccw_fast_01_01.mat',
    '/Volumes/LPM02 storage/Datasets/Bio/Camargo_2021/AB06/10_09_18/ramp/conditions/ramp_1_l_01_01.mat',
    '/Volumes/LPM02 storage/Datasets/Bio/Camargo_2021/AB06/10_09_18/stair/conditions/stair_1_l_01_01.mat',
    '/Volumes/LPM02 storage/Datasets/Bio/Camargo_2021/AB06/10_09_18/treadmill/conditions/treadmill_01_01.mat',
]

for path in paths:
    mode = path.split('/')[-3]
    d = sio.loadmat(path)
    keys = [k for k in d.keys() if not k.startswith('__') and k != 'None']
    print(f"{mode}: {keys}")
    print()

levelground: ['subject', 'file', 'trialType', 'leadingLegStart', 'leadingLegStop', 'labelError', 'trialStarts', 'trialEnds', 'turn', 'speed']

ramp: ['subject', 'file', 'trialStarts', 'trialEnds', 'transLegAscent', 'transLegDescent', 'rampIncline']

stair: ['subject', 'file', 'trialStarts', 'trialEnds', 'transLegAscent', 'transLegDescent', 'stairHeight']

treadmill: ['trialStarts', 'trialEnds']



### Conditions File by Mode

| Key | levelground | treadmill | stair | ramp |
|---|---|---|---|---|
| `trialStarts` / `trialEnds` | ✓ | ✓ | ✓ | ✓ |
| `speed` | ✓ | | | |
| `stairHeight` | | | ✓ | |
| `rampIncline` | | | | ✓ |
| `transLegAscent/Descent` | | | ✓ | ✓ |
| `leadingLegStart/Stop` | ✓ | | | |
| `turn` | ✓ | | | |
| `labelError` | ✓ | | | |

Notes:
+ Treadmill is the sparsest — speed is encoded in the filename only
+ Mode-aware parsing will be needed when building the data loader

In [5]:
# Open IMU file(s) to inspect contents
from pymatreader import read_mat

imu_path = "/Volumes/LPM02 storage/Datasets/Bio/Camargo_2021/AB06/10_09_18/levelground/imu/levelground_ccw_fast_01_01.mat"

imu = read_mat(imu_path)
print(imu.keys())
print(imu['None'])

dict_keys(['__header__', '__version__', '__globals__', 'None', '__function_workspace__'])
{'s0': [b'data'], 's1': [b'MCOS'], 's2': [b'table'], 'arr': [array([3707764736,          2,          1,          1,          1,
                1], dtype=uint32)]}


## Dataset Change — Camargo to PAMAP2

+ Camargo dataset stores all sensor data as MATLAB table objects
+ Requires MATLAB or Octave installation 
+ Pivot to PAMAP2 dataset, which provides IMU data in format compatible with pandas.

This notebook is retained as a record of the data loading investigation.